# 🧹 SEC EDGAR — Data Cleaning & Validation
**Project:** SEC EDGAR Financial Ratio Analysis  
**Engineer:** Meet Saini  
**Last Updated:** 2026-05-30

---

### Notebook Objectives
1. Clean and cast column data types
2. Handle empty strings and nulls in value column
3. Filter segment-level rows from edgar_num_all
4. Validate cleaned tables against master tables
5. Produce two clean tables ready for SQL analysis

### Input Tables
- `edgar_sub_all` → 192,059 rows
- `edgar_num_all` → 88,170,247 rows

### Output Tables
- `edgar_sub_clean` → cleaned filing metadata
- `edgar_num_clean` → cleaned financial numbers

## Step 1 — Environment Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, to_date

spark = SparkSession.builder.getOrCreate()
print(f"✅ Spark session ready | Version: {spark.version}")

## Step 2 — Pre-Cleaning Row Counts
Baseline counts before cleaning — used for validation after cleaning.

In [0]:
sub_count = spark.table("edgar_sub_all").count()
num_count = spark.table("edgar_num_all").count()

print(f"{'='*45}")
print(f"PRE-CLEANING COUNTS")
print(f"{'='*45}")
print(f"edgar_sub_all : {sub_count:,} rows")
print(f"edgar_num_all : {num_count:,} rows")
print(f"{'='*45}")

## Step 3 — Clean `edgar_sub_all`

Cleaning steps:
- Cast `filed` from string to date
- Cast `sic` from string to integer
- Trim whitespace from `name` and `countryba`
- Save as `edgar_sub_clean`

In [0]:
sub_df = spark.table("edgar_sub_all")

sub_clean = sub_df \
    .withColumn("filed", to_date(col("filed"), "yyyyMMdd")) \
    .withColumn("sic", 
        when(trim(col("sic")) == "", None)
        .otherwise(col("sic").cast("integer"))
    ) \
    .withColumn("name", trim(col("name"))) \
    .withColumn("countryba", trim(col("countryba")))

sub_clean.write.format("delta").mode("overwrite").saveAsTable("edgar_sub_clean")

print(f"✅ edgar_sub_clean saved | Rows: {sub_clean.count():,}")

## Step 4 — Clean `edgar_num_all`

Cleaning steps:
- Filter out segment-level rows (keep segments IS NULL only)
- Replace empty strings in `value` with NULL
- Cast `value` from string to DOUBLE using TRY_CAST
- Cast `ddate` from string to date
- Remove rows where value is NULL after casting
- Save as `edgar_num_clean`

In [0]:
num_df = spark.table("edgar_num_all")

num_clean = num_df \
    .filter(trim(col("segments")) == "") \
    .withColumn("value",
        when(trim(col("value")) == "", None)
        .otherwise(col("value"))
    ) \
    .withColumn("value", col("value").cast("double")) \
    .withColumn("ddate", to_date(col("ddate"), "yyyyMMdd")) \
    .filter(col("value").isNotNull())

num_clean.write.format("delta").mode("overwrite").saveAsTable("edgar_num_clean")

print(f"✅ edgar_num_clean saved | Rows: {num_clean.count():,}")

## Step 5 — Post-Cleaning Validation

Validates:
- Row counts before and after cleaning
- No nulls in critical columns
- Data types are correctly cast
- Value column has no empty strings

In [0]:
sub_clean_count = spark.table("edgar_sub_clean").count()
num_clean_count = spark.table("edgar_num_clean").count()

print(f"{'='*55}")
print(f"POST-CLEANING VALIDATION")
print(f"{'='*55}")
print(f"{'Table':<25} {'Before':>12} {'After':>12} {'Dropped':>12}")
print(f"{'-'*55}")
print(f"{'edgar_sub_all':<25} {sub_count:>12,} {sub_clean_count:>12,} {sub_count - sub_clean_count:>12,}")
print(f"{'edgar_num_all':<25} {num_count:>12,} {num_clean_count:>12,} {num_count - num_clean_count:>12,}")
print(f"{'='*55}")

## Step 6 — Schema Validation
Confirms data types are correctly cast after cleaning.

In [0]:
print("=== edgar_sub_clean SCHEMA ===")
spark.table("edgar_sub_clean").printSchema()

print("\n=== edgar_num_clean SCHEMA ===")
spark.table("edgar_num_clean").printSchema()

## Step 7 — Null Check on Cleaned Tables
Confirms no critical nulls remain after cleaning.

In [0]:
from pyspark.sql.functions import sum as spark_sum, when

def count_nulls(table_name):
    df = spark.table(table_name)
    
    null_exprs = [] 
    for c in df.columns:
        expr = spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        null_exprs.append(expr)
    
    null_counts = df.select(null_exprs).collect()[0].asDict()
    
    print(f"\nNULL CHECK — {table_name}")
    print("-" * 40)
    for col_name, null_count in sorted(null_counts.items(), key=lambda x: -x[1]):
        if null_count > 0:
            print(f"⚠️  {col_name}: {null_count:,} nulls")
    print("✅ Done")

count_nulls("edgar_sub_clean")
count_nulls("edgar_num_clean")

## Step 8 — Sample Preview of Cleaned Tables
Final sanity check on cleaned data.

In [0]:
print("=== edgar_sub_clean PREVIEW ===")
spark.sql("""
    SELECT adsh, cik, name, sic, filed, period, source_year
    FROM edgar_sub_clean
    LIMIT 5
""").show(truncate=False)

print("=== edgar_num_clean PREVIEW ===")
spark.sql("""
    SELECT adsh, tag, ddate, qtrs, uom, value, period, source_year
    FROM edgar_num_clean
    LIMIT 5
""").show(truncate=False)